Import données Infrabel

In [2]:
#importer les librairies
import requests
from tqdm import tqdm
import os
import pandas as pd


Liste des datasets à importer depuis l'Open Data d'Infrabel

In [3]:
# premier dataset pour tester mais le dictionnaire final sera beaucoup plus long
datasets = {
    "points_ops" : "https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets/operationele-punten-van-het-netwerk/exports/parquet?lang=fr&timezone=Europe%2FBerlin"
}

Fonction pour télécharger les datasets

In [4]:
#Fonction pour télécharger les datasets
def download_dataset(dataset_url, output_path, dataset_name, chunk_size=10240):
    print(f"Début du téléchargement : {dataset_name}")
    try: 
        # téléchager le dataset et mettre un "timer" pour contrôler le temps de connection au serveur (10)
            # et le temps de réception entre chaque chunk (120)
        response = requests.get(dataset_url, stream=True, timeout=(10, 120))
        response.raise_for_status()
        
        # récupérer la taille du dataset
        total_size = int(response.headers.get("content-length", 0))
        
        # gérer les datasets créés au moment du téléchargement
        total_to_display = total_size if total_size > 0 else None

        # vérifier que le répertoire data/raw/ existe et le créer s'il n'existe pas
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        # chunker le dataset pour gérer sa taille
        with open(output_path, "wb") as f:
            # wrapper la boucle dans tqdm pour afficher une barre de progression
            with tqdm(
                total=total_to_display,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
                desc=f"Downloading dataset {dataset_name}"
                ) as pbar:
                    # écrire le fichier et faire avancer la barre de progression
                    for chunk in response.iter_content(chunk_size):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
            print(f"{dataset_name} téléchargé dans {output_path}")
    
    except requests.exceptions.RequestException as error: 
            print(f"Erreur lors du téléchargement de {dataset_name} : {error}")

   

Pour télécharger tous les datasets Infrabel

In [5]:
# exécuter la fonction sur le dictionnaire de datasets
# par défaut, la taille des chunks est de 10240
for name, url in datasets.items():
    output_path = f"data/raw/{name}.parquet"
    dataset_url = url
    dataset_name = name
    download_dataset(dataset_url, output_path, dataset_name)

Début du téléchargement : points_ops


points_ops téléchargé dans data/raw/points_ops.parquet


Vérifier que le fichier .parquet est bien téléchargé

In [6]:
df = pd.read_parquet("data/raw/points_ops.parquet")
df.head()

,geo_point_2d,geo_shape,ptcarid,taftapcode,symbolicname,shortnamefrench,shortnamedutch,longnamefrench,longnamedutch,commercialshortnamefrench,commercialshortnamedutch,commercialmiddlenamefrench,commercialmiddlenamedutch,commerciallongnamefrench,commerciallongnamedutch,classification,class_en,class_fr
0,b'\x01\x01\x00\x00\x00\x13\x11\xbf\xc37\xfd\x1...,b'\x01\x01\x00\x00\x00\x13\x11\xbf\xc37\xfd\x1...,443,BE00443,FKGLF,GENK-ZD-FORD,GENK-ZD-FORD,GENK-ZUID-L.O.-FORD,GENK-ZUID-L.O.-FORD,Genk-Zd-Ford,Genk-Zd-Ford,Genk-Zuid-L.O.-Ford,Genk-Zuid-L.O.-Ford,Genk-Zuid-L.O.-Ford,Genk-Zuid-L.O.-Ford,Station,Station,Gare
1,b'\x01\x01\x00\x00\x00\r\to\xbe\xd6\xd5\x11@\x...,b'\x01\x01\x00\x00\x00\r\to\xbe\xd6\xd5\x11@\x...,275,BE00275,GCRA,CHARLEROI-AT,CHARLEROI-AT,CHARLEROI-A.T.,CHARLEROI-A.T.,Charleroi-AT,Charleroi-AT,Charleroi-A.T.,Charleroi-A.T.,Charleroi-A.T.,Charleroi-A.T.,Dienstinstallatie,Service installation,Installation de service
2,"b'\x01\x01\x00\x00\x00>\x84\x92P\xf8,\x11@\x15...","b'\x01\x01\x00\x00\x00>\x84\x92P\xf8,\x11@\x15...",1748,BE01748,VOIL1,V.OILTANK1,V.OILTANK1,VERB.OILTANKING 1,VERB.OILTANKING 1,V.Oiltank1,V.Oiltank1,Verb.Oiltanking 1,Verb.Oiltanking 1,Verb.Oiltanking 1,Verb.Oiltanking 1,Verbinding,Connection,Raccordement
3,b'\x01\x01\x00\x00\x00/J\x7f\xef4)\x11@o\xf1\x...,b'\x01\x01\x00\x00\x00/J\x7f\xef4)\x11@o\xf1\x...,1749,BE01749,VOIL2,V.OILTANK2,V.OILTANK2,VERB.OILTANKING 2,VERB.OILTANKING 2,V.Oiltank2,V.Oiltank2,Verb.Oiltanking 2,Verb.Oiltanking 2,Verb.Oiltanking 2,Verb.Oiltanking 2,Verbinding,Connection,Raccordement
4,"b'\x01\x01\x00\x00\x00\x93\xbc\xb4\xbeF""\x11@7...","b'\x01\x01\x00\x00\x00\x93\xbc\xb4\xbeF""\x11@7...",1751,BE01751,VIBR,V.IBR,V.IBR,VERB.IBR,VERB.IBR,V.IBR,V.IBR,Verb.IBR,Verb.IBR,Verb.IBR,Verb.IBR,Verbinding,Connection,Raccordement
